<a href="https://colab.research.google.com/github/adhishmodi/customer-churn-prediction/blob/feature/milestone-3-model-training/notebooks/03_model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Customer Churn Prediction — Model Training

Train and compare classical machine-learning models for the IBM Telco Customer Churn dataset. This milestone uses **cross-validation on the training split only**. The held-out test split is reserved for Milestone 4 and must not influence model selection or hyperparameter tuning.

**Primary selection metric:** ROC-AUC. We also report Average Precision, Precision, Recall, F1, and Accuracy. Average Precision is useful because churn is the minority class.

Every candidate is wrapped in a pipeline containing preprocessing. Imputation, scaling, and one-hot encoding are therefore learned separately inside each cross-validation training fold.

## 1. Colab setup and dataset upload

Run this notebook from top to bottom. The raw CSV is not stored in GitHub; upload the original `WA_Fn-UseC_-Telco-Customer-Churn.csv` when prompted.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPOSITORY_URL = "https://github.com/adhishmodi/customer-churn-prediction.git"
GIT_BRANCH = "feature/milestone-3-model-training"
EXPECTED_CSV = "WA_Fn-UseC_-Telco-Customer-Churn.csv"
current = Path.cwd()
if not (current / "src" / "train.py").exists():
    project = Path("/content/customer-churn-prediction")
    if not project.exists():
        subprocess.run(["git", "clone", "--branch", GIT_BRANCH, REPOSITORY_URL, str(project)], check=True)
    %cd /content/customer-churn-prediction
else:
    project = current

DATA_PATH = project / "data" / "raw" / EXPECTED_CSV
DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
if not DATA_PATH.exists():
    try:
        from google.colab import files
        print(f"Upload {EXPECTED_CSV}")
        uploaded = files.upload()
    except ImportError as exc:
        raise FileNotFoundError(f"Place the dataset at {DATA_PATH}") from exc
    if EXPECTED_CSV not in uploaded:
        csv_names = [name for name in uploaded if name.lower().endswith(".csv")]
        if len(csv_names) != 1:
            raise ValueError(f"Upload exactly one CSV named {EXPECTED_CSV}. Found: {csv_names}")
        selected_name = csv_names[0]
    else:
        selected_name = EXPECTED_CSV
    DATA_PATH.write_bytes(uploaded[selected_name])
assert DATA_PATH.exists(), "Dataset upload did not create the expected file."
print(f"Project directory: {project}")
print(f"Dataset: {DATA_PATH}")

## 2. Load, clean, and validate the data

Use the deterministic cleaning and fixed feature list from Milestone 2. `customerID` is excluded and `Churn` is encoded as 0/1. No learned preprocessing happens here.

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

from src.preprocess import load_and_clean_data, prepare_features_target, split_data
from src.train import (
    compare_models, get_best_model, save_candidate_model,
    save_comparison_report, save_training_metadata, tune_models,
)
from src.config import MODELS_DIR, RANDOM_SEED, REPORTS_DIR

raw_df = pd.read_csv(DATA_PATH)
cleaned_df = load_and_clean_data(DATA_PATH)
X, y = prepare_features_target(cleaned_df)
assert "customerID" not in X.columns and "Churn" not in X.columns
assert set(y.unique()) == {0, 1}
print(f"Rows: {len(X):,} | Raw columns: {raw_df.shape[1]} | Model input features: {X.shape[1]}")
print(f"Churn rate: {y.mean():.2%}")
display(y.value_counts().rename(index={0: "No churn", 1: "Churn"}).to_frame("customers"))

## 3. Recreate the reproducible train/test split

Use the same 80/20 stratified split and random seed (42) from Milestone 2. The test variables are kept aside and will not be passed to comparison, tuning, or selection functions.

In [ ]:
X_train, X_test_reserved, y_train, y_test_reserved = split_data(X, y)
print(f"Training rows: {len(X_train):,}")
print(f"Reserved test rows: {len(X_test_reserved):,} (not used in this milestone)")
print(f"Training churn rate: {y_train.mean():.2%}")
assert X_train.index.intersection(X_test_reserved.index).empty
assert y_train.index.equals(X_train.index)
assert y_test_reserved.index.equals(X_test_reserved.index)

## 4. Evaluation strategy and candidate models

We use five-fold `StratifiedKFold`. Each fold preserves the churn/non-churn proportion. The primary selection metric is ROC-AUC; Average Precision complements it by focusing on the positive (churn) class. Accuracy alone can be misleading when classes are imbalanced.

The initial candidates are a Dummy baseline, Logistic Regression, Decision Tree, Random Forest, and XGBoost. The Dummy model provides a minimum reference point; the other models learn patterns from customer features.

In [ ]:
from sklearn.model_selection import StratifiedKFold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
print(cv)
print("Selection metric: mean cross-validation ROC-AUC")
print("Secondary metrics: Average Precision, Precision, Recall, F1, Accuracy")

## 5. Initial model comparison

Each model is evaluated using raw `X_train` and `y_train`. The preprocessing pipeline is fitted independently within every fold. This avoids leakage from validation folds into imputation, scaling, or category encoding.

In [ ]:
initial_results = compare_models(X_train, y_train)
initial_report = save_comparison_report(
    initial_results, REPORTS_DIR / "initial_model_comparison.csv"
)
display(initial_report.round(4))

## 6. Hyperparameter tuning

We tune Logistic Regression, Random Forest, and XGBoost with `RandomizedSearchCV`. Each search samples ten configurations by default and uses the same five-fold stratified CV. The search refits the best configuration using the full training split. The held-out test split is never supplied to the search.

The search spaces are intentionally compact so the experiment remains practical in Colab. You can lower `N_ITER` for a quick smoke run, but record the value used in your final report.

In [ ]:
N_ITER = 10
tuned_results = tune_models(X_train, y_train, n_iter=N_ITER)
tuned_rows = []
for model_name, result in tuned_results.items():
    tuned_rows.append({"model": model_name, **result["cv_metrics"], "best_params": result["best_params"]})
tuned_summary = pd.DataFrame(tuned_rows).sort_values("cv_roc_auc_mean", ascending=False)
display(tuned_summary.round(4))

## 7. Select the best candidate using cross-validation only

Combine the initial and tuned candidates, then select the highest mean CV ROC-AUC. Do not compare models using test-set scores. The selected pipeline contains both the preprocessing transformer and classifier.

In [ ]:
all_results = {**initial_results, **tuned_results}
best_name, best_pipeline = get_best_model(all_results, metric="roc_auc")
best_result = all_results[best_name]
print(f"CV-selected candidate: {best_name}")
print(f"Mean CV ROC-AUC: {best_result['cv_metrics']['cv_roc_auc_mean']:.4f}")
print(f"CV ROC-AUC standard deviation: {best_result['cv_metrics']['cv_roc_auc_std']:.4f}")
print("No held-out test metrics have been calculated.")

## 8. Fit and save the model candidate

If an initial (untuned) model wins, fit its complete pipeline on all training rows. A tuned candidate is refitted by `RandomizedSearchCV`; we fit it explicitly for a consistent final artifact. The fitted preprocessing steps are saved together with the classifier.

In [ ]:
best_pipeline.fit(X_train, y_train)
model_path = MODELS_DIR / "best_model_candidate.joblib"
save_candidate_model(best_pipeline, model_path)
save_training_metadata(
    model_name=best_name, cv_metrics=best_result["cv_metrics"],
    best_params=best_result.get("best_params", {}),
    path=MODELS_DIR / "training_metadata.json",
)
final_report = save_comparison_report(all_results, REPORTS_DIR / "model_comparison.csv")
print(f"Saved model candidate: {model_path}")
print(f"Saved comparison report: {REPORTS_DIR / 'model_comparison.csv'}")
print(f"Saved metadata: {MODELS_DIR / 'training_metadata.json'}")
display(final_report.round(4))

## 9. Final checks and Milestone 4 hand-off

The selected artifact should reload successfully and contain a fitted preprocessing pipeline. Generated files live in ignored `models/` and `reports/` folders; do not commit the dataset or generated artifacts.

**Important:** this milestone ends with a CV-selected model candidate. The held-out test split has not been evaluated. Milestone 4 will perform the final test evaluation once, after the model and threshold strategy are frozen.

In [ ]:
import joblib
reloaded_model = joblib.load(model_path)
assert hasattr(reloaded_model.named_steps["preprocessor"], "transformers_")
assert hasattr(reloaded_model.named_steps["classifier"], "classes_")
assert not final_report.empty
assert "test_roc_auc" not in final_report.columns
print("Model artifact reload check passed.")
print("CV comparison contains", len(final_report), "candidate rows.")
print("Held-out test evaluation remains reserved for Milestone 4.")